# Stage 1: Extract Monthly BTS Files for 2025

This notebook starts the **Extract** stage of the Flight Reliability Intelligence ETL process.

**What we will do in this notebook**
1. Locate the 12 monthly BTS CSV files for 2025.
2. Verify that all 12 expected monthly files exist.
3. Verify that the files have compatible column structures.
4. Read the 12 monthly files.
5. Combine them into one DataFrame that represents all flights in 2025.
6. Export the combined DataFrame to `data/processed/flights_2025.csv`.

**What we will not do yet**
- We will not change data types.
- We will not handle missing values.
- We will not remove duplicates.
- We will not clean, replace, fill, or otherwise modify the data.

The goal of this notebook is only to create, verify, and export the yearly 2025 DataFrame. After that, we will stop and inspect data types and missing values in the next stage.

## 1. Import libraries

We import `pandas` to read and combine the CSV files, and `pathlib` to work with file paths in a clear, readable way.

In [1]:
from pathlib import Path

import pandas as pd


## 2. Define the raw data folder

The monthly BTS files are stored in `data/raw/bts/2025/`.

This cell finds the project root whether the notebook is run from the project root or from the `notebooks/` folder. That makes the path easier to reuse later in a Python script.

In [2]:
def get_project_root():
    """Return the project root folder.

    The notebook may be run from the project root or from notebooks/.
    """
    current_folder = Path.cwd()

    if current_folder.name == "notebooks":
        return current_folder.parent

    return current_folder


PROJECT_ROOT = get_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "bts" / "2025"

print("Project root:")
print(PROJECT_ROOT)
print()
print("Raw data folder:")
print(RAW_DATA_DIR)
print()
print("Folder exists:", RAW_DATA_DIR.exists())


Project root:
/home/gur/Documents/Protfolio/Flight Reliability Intelligence System/Flight-Reliability-Intelligence

Raw data folder:
/home/gur/Documents/Protfolio/Flight Reliability Intelligence System/Flight-Reliability-Intelligence/data/raw/bts/2025

Folder exists: True


## 3. Locate the 12 monthly CSV files

We look for files named `2025_01.csv` through `2025_12.csv`.

This step only lists the files. It does not read their contents yet.

In [3]:
EXPECTED_FILE_NAMES = [f"2025_{month:02d}.csv" for month in range(1, 13)]

csv_files = sorted(RAW_DATA_DIR.glob("2025_*.csv"))
found_file_names = [file_path.name for file_path in csv_files]

print("Expected files:")
for file_name in EXPECTED_FILE_NAMES:
    print(f"- {file_name}")

print()
print("Files found in the folder:")
for file_path in csv_files:
    print(f"- {file_path.name}")


Expected files:
- 2025_01.csv
- 2025_02.csv
- 2025_03.csv
- 2025_04.csv
- 2025_05.csv
- 2025_06.csv
- 2025_07.csv
- 2025_08.csv
- 2025_09.csv
- 2025_10.csv
- 2025_11.csv
- 2025_12.csv

Files found in the folder:
- 2025_01.csv
- 2025_02.csv
- 2025_03.csv
- 2025_04.csv
- 2025_05.csv
- 2025_06.csv
- 2025_07.csv
- 2025_08.csv
- 2025_09.csv
- 2025_10.csv
- 2025_11.csv
- 2025_12.csv


## 4. Verify that all 12 expected monthly files exist

Before we read any data, we confirm that:

- All 12 expected monthly files are present.
- There are no unexpected extra CSV files in this folder.

If a file is missing, we should stop and fix the data folder before continuing.

In [4]:
missing_files = [file_name for file_name in EXPECTED_FILE_NAMES if file_name not in found_file_names]
extra_files = [file_name for file_name in found_file_names if file_name not in EXPECTED_FILE_NAMES]

print("Number of expected files:", len(EXPECTED_FILE_NAMES))
print("Number of files found:", len(found_file_names))
print()

if missing_files:
    print("Missing files:")
    for file_name in missing_files:
        print(f"- {file_name}")
else:
    print("No missing files.")

print()

if extra_files:
    print("Unexpected extra files:")
    for file_name in extra_files:
        print(f"- {file_name}")
else:
    print("No unexpected extra files.")

print()

if missing_files:
    raise FileNotFoundError(
        "One or more expected monthly CSV files are missing. "
        "Please add the missing files before continuing."
    )

print("All 12 expected monthly files are present.")


Number of expected files: 12
Number of files found: 12

No missing files.

No unexpected extra files.

All 12 expected monthly files are present.


## 5. Verify that the file structures are compatible

We expect every monthly file to have the same columns, in the same order.

This check reads **only the header row** of each file. It does not load the flight records yet.

If the columns do not match, combining the files would be unsafe, so we stop and inspect the difference.

In [5]:
def get_csv_columns(file_path):
    """Read only the column names from a CSV file."""
    header_df = pd.read_csv(file_path, nrows=0)
    return list(header_df.columns)


file_columns = {}

for file_path in csv_files:
    file_columns[file_path.name] = get_csv_columns(file_path)
    print(f"{file_path.name}: {len(file_columns[file_path.name])} columns")

reference_file_name = csv_files[0].name
reference_columns = file_columns[reference_file_name]

print()
print("Reference file:", reference_file_name)
print("Reference column count:", len(reference_columns))
print()
print("Column names:")
for column_name in reference_columns:
    print(f"- {column_name}")


2025_01.csv: 62 columns
2025_02.csv: 62 columns
2025_03.csv: 62 columns
2025_04.csv: 62 columns
2025_05.csv: 62 columns
2025_06.csv: 62 columns
2025_07.csv: 62 columns
2025_08.csv: 62 columns
2025_09.csv: 62 columns
2025_10.csv: 62 columns
2025_11.csv: 62 columns
2025_12.csv: 62 columns

Reference file: 2025_01.csv
Reference column count: 62

Column names:
- YEAR
- QUARTER
- MONTH
- DAY_OF_MONTH
- DAY_OF_WEEK
- FL_DATE
- OP_UNIQUE_CARRIER
- OP_CARRIER_AIRLINE_ID
- OP_CARRIER
- TAIL_NUM
- OP_CARRIER_FL_NUM
- ORIGIN_AIRPORT_ID
- ORIGIN_AIRPORT_SEQ_ID
- ORIGIN_CITY_MARKET_ID
- ORIGIN
- ORIGIN_CITY_NAME
- ORIGIN_STATE_ABR
- ORIGIN_STATE_NM
- ORIGIN_WAC
- DEST_AIRPORT_ID
- DEST_AIRPORT_SEQ_ID
- DEST_CITY_MARKET_ID
- DEST
- DEST_CITY_NAME
- DEST_STATE_ABR
- DEST_STATE_NM
- DEST_WAC
- CRS_DEP_TIME
- DEP_TIME
- DEP_DELAY
- DEP_DELAY_NEW
- DEP_DEL15
- DEP_TIME_BLK
- TAXI_OUT
- WHEELS_OFF
- WHEELS_ON
- TAXI_IN
- CRS_ARR_TIME
- ARR_TIME
- ARR_DELAY
- ARR_DELAY_NEW
- ARR_DEL15
- ARR_TIME_BLK
- CAN

This cell compares every file against the first monthly file.

We check both the column names and the column order. If they match, the files can be combined safely.

In [6]:
structure_mismatches = []

for file_name, columns in file_columns.items():
    if columns != reference_columns:
        structure_mismatches.append(file_name)
        print(f"Structure mismatch: {file_name}")
        print(f"  Column count: {len(columns)} (reference has {len(reference_columns)})")

        # Show only the differences, so the output stays easy to read.
        missing_columns = [col for col in reference_columns if col not in columns]
        extra_columns = [col for col in columns if col not in reference_columns]

        if missing_columns:
            print("  Missing columns:", missing_columns)
        if extra_columns:
            print("  Extra columns:", extra_columns)
        if not missing_columns and not extra_columns:
            print("  Column names are the same, but the order is different.")

if structure_mismatches:
    raise ValueError(
        "The monthly CSV files do not all have the same structure. "
        "Please inspect the mismatched files before combining them."
    )

print()
print("All 12 monthly files have the same column names and the same column order.")
print("The files are compatible and can be combined.")



All 12 monthly files have the same column names and the same column order.
The files are compatible and can be combined.


## 6. Read the 12 monthly CSV files

Now that the files exist and have matching structures, we read each monthly CSV into its own DataFrame.

We keep the data as pandas reads it. We do not convert types, fill missing values, or drop rows.

This step can take a few minutes because the files are large.

In [7]:
monthly_frames = []
monthly_row_counts = {}

for file_path in csv_files:
    print(f"Reading {file_path.name}...")
    month_df = pd.read_csv(file_path, low_memory=False)
    monthly_frames.append(month_df)
    monthly_row_counts[file_path.name] = len(month_df)
    print(f"  Rows: {len(month_df):,}")
    print(f"  Columns: {len(month_df.columns)}")

print()
print("Monthly row counts:")
for file_name, row_count in monthly_row_counts.items():
    print(f"- {file_name}: {row_count:,}")

total_monthly_rows = sum(monthly_row_counts.values())
print()
print("Total rows across all monthly files:", f"{total_monthly_rows:,}")


Reading 2025_01.csv...
  Rows: 539,747
  Columns: 62
Reading 2025_02.csv...
  Rows: 504,884
  Columns: 62
Reading 2025_03.csv...
  Rows: 600,872
  Columns: 62
Reading 2025_04.csv...
  Rows: 583,950
  Columns: 62
Reading 2025_05.csv...
  Rows: 605,648
  Columns: 62
Reading 2025_06.csv...
  Rows: 611,575
  Columns: 62
Reading 2025_07.csv...
  Rows: 631,428
  Columns: 62
Reading 2025_08.csv...
  Rows: 602,378
  Columns: 62
Reading 2025_09.csv...
  Rows: 562,439
  Columns: 62
Reading 2025_10.csv...
  Rows: 605,844
  Columns: 62
Reading 2025_11.csv...
  Rows: 570,550
  Columns: 62
Reading 2025_12.csv...
  Rows: 582,304
  Columns: 62

Monthly row counts:
- 2025_01.csv: 539,747
- 2025_02.csv: 504,884
- 2025_03.csv: 600,872
- 2025_04.csv: 583,950
- 2025_05.csv: 605,648
- 2025_06.csv: 611,575
- 2025_07.csv: 631,428
- 2025_08.csv: 602,378
- 2025_09.csv: 562,439
- 2025_10.csv: 605,844
- 2025_11.csv: 570,550
- 2025_12.csv: 582,304

Total rows across all monthly files: 7,001,619


## 7. Combine the monthly files into one 2025 DataFrame

We stack the 12 monthly DataFrames into one DataFrame named `flights_2025`.

This is a vertical combination: January rows, then February rows, and so on, until December. No columns are added or removed.

In [8]:
flights_2025 = pd.concat(monthly_frames, ignore_index=True)

# The monthly frames are no longer needed after the yearly DataFrame is created.
del monthly_frames

print("Yearly DataFrame created.")
print("Name: flights_2025")
print("Rows:", f"{len(flights_2025):,}")
print("Columns:", len(flights_2025.columns))


Yearly DataFrame created.
Name: flights_2025
Rows: 7,001,619
Columns: 62


## 8. Verify the yearly DataFrame

Before we stop, we confirm that the combination worked as expected:

- The yearly row count matches the sum of the monthly row counts.
- The column names are unchanged.
- All 12 months are present.
- The year value is 2025.

These checks only inspect the data. They do not change it.

In [9]:
print("Yearly DataFrame shape:", flights_2025.shape)
print("Expected rows:", f"{total_monthly_rows:,}")
print("Actual rows:", f"{len(flights_2025):,}")
print()

if len(flights_2025) != total_monthly_rows:
    raise ValueError("The yearly row count does not match the sum of the monthly row counts.")

print("Row count check passed.")
print()

if list(flights_2025.columns) != reference_columns:
    raise ValueError("The yearly DataFrame columns do not match the original CSV columns.")

print("Column structure check passed.")
print()

print("Years in the data:")
print(sorted(flights_2025["YEAR"].unique()))
print()

print("Months in the data:")
print(sorted(flights_2025["MONTH"].unique()))
print()

print("Row count by month:")
print(flights_2025["MONTH"].value_counts().sort_index())


Yearly DataFrame shape: (7001619, 62)
Expected rows: 7,001,619
Actual rows: 7,001,619

Row count check passed.

Column structure check passed.

Years in the data:
[2025]

Months in the data:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Row count by month:
MONTH
1     539747
2     504884
3     600872
4     583950
5     605648
6     611575
7     631428
8     602378
9     562439
10    605844
11    570550
12    582304
Name: count, dtype: int64


We also display a small sample from the beginning and the end of the yearly DataFrame. This helps confirm that the files were stacked correctly, from January through December.

In [11]:
print("First 5 rows:")
display(flights_2025.head())

print()
print("Last 5 rows:")
display(flights_2025.tail())


First 5 rows:


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,...,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS,DIV_REACHED_DEST,DIV_ACTUAL_ELAPSED_TIME,DIV_ARR_DELAY,DIV_DISTANCE,DIV1_AIRPORT,DIV1_AIRPORT_ID
0,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N101NN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N101NN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N102UW,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N103NN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,1,3,1/1/2025 12:00:00 AM,AA,19805,AA,N103NN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Last 5 rows:


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,...,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS,DIV_REACHED_DEST,DIV_ACTUAL_ELAPSED_TIME,DIV_ARR_DELAY,DIV_DISTANCE,DIV1_AIRPORT,DIV1_AIRPORT_ID
7001614,2025,4,12,31,3,12/31/2025 12:00:00 AM,YX,20452,YX,N880RW,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
7001615,2025,4,12,31,3,12/31/2025 12:00:00 AM,YX,20452,YX,N880RW,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
7001616,2025,4,12,31,3,12/31/2025 12:00:00 AM,YX,20452,YX,N882RW,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
7001617,2025,4,12,31,3,12/31/2025 12:00:00 AM,YX,20452,YX,N979RP,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
7001618,2025,4,12,31,3,12/31/2025 12:00:00 AM,YX,20452,YX,N979RP,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## 9. Export the combined yearly dataset

We save the combined DataFrame to `data/processed/flights_2025.csv`.

This file is a snapshot of all 2025 flights after the 12 monthly files have been combined and verified. It is not a cleaned dataset. We export it so later steps can start from the yearly file instead of reading the 12 monthly files again.

The pandas index is not written to the CSV, because it is not part of the original BTS data.


In [12]:
# Save the combined yearly data under data/processed/.
processed_dir = PROJECT_ROOT / "data" / "processed"
output_file = processed_dir / "flights_2025.csv"

# Create the folder if it does not already exist.
processed_dir.mkdir(parents=True, exist_ok=True)

# Export the DataFrame as it currently exists. Do not add the pandas index.
flights_2025.to_csv(output_file, index=False)

print("Export complete.")
print("Output file:", output_file.relative_to(PROJECT_ROOT))
print("Rows exported:", f"{len(flights_2025):,}")


Export complete.
Output file: data/processed/flights_2025.csv
Rows exported: 7,001,619


## Stage 1 complete

The Extract stage now has one DataFrame, `flights_2025`, that contains all monthly BTS flight records for 2025. The same data has been exported to `data/processed/flights_2025.csv`.

**What we verified**
- All 12 expected monthly files exist.
- All 12 files have the same column structure.
- The files were combined without changing column names or row counts.
- The combined dataset was exported without adding a pandas index.

**Next stage**
We will inspect data types and missing values together before making any cleaning or transformation decisions.
